# Desafio RPA

## Introdução
Este notebook apresenta uma solução ETL automatizada com RPA para coleta de filmes no portal [RPA Challenge](https://rpachallenge.com/).

## Objetivo
Construir e demonstrar um pipeline completo de **Extract, Transform, Load (ETL)** que:

1. Extrai filmes da aba Movie Search para a busca `Avengers`.
2. Transforma os dados para consistência e deduplicação.
3. Carrega os resultados em MySQL com controle de duplicidade.
4. Gera outputs rastreáveis (CSV, JSON e dump SQL).

## Arquitetura da Solução

Fluxo lógico do pipeline:

```mermaid
flowchart LR
    A[Selenium RPA\nRPA Challenge - Movie Search] --> B[Extract\nColeta nome e descrição completa]
    B --> C[Transform\nLimpeza, padronização, deduplicação]
    C --> D[Load\nMySQL upsert]
    C --> E[Outputs\nCSV e JSON]
    C --> F[Dump SQL\nreexecução e auditoria]
```

### Estrutura de Projeto

- `wellbe_rpa_pipeline.ipynb`: relatório técnico e execução guiada.
- `src/scraper.py`: automação Selenium com waits explícitos.
- `src/database.py`: conexão MySQL, schema e upsert.
- `src/utils.py`: funções utilitárias de transformação e output.
- `src/config.py`: configurações via variáveis de ambiente.
- `src/pipeline.py`: orquestração da execução fim a fim.
- `sql/create_movies_table.sql`: script de criação da tabela.
- `outputs/`: arquivos finais CSV e JSON.
- `data/`: camada de persistência intermediária (raw).

## Tecnologias Utilizadas

- **Python 3**: linguagem principal do pipeline.
- **Selenium**: automação RPA para navegacao e scraping.
- **Pandas / Numpy**: transformação de dados tabulares.
- **MySQL**: persistência relacional com upsert.
- **Jupyter Notebook**: documentação técnica e reproducibilidade.

## Etapas do Pipeline (ETL)

- **Extract**: navega no site, acessa Movie Search, busca `Avengers` e captura nome + descrição completa via `card-reveal`.
- **Transform**: remove vazios, normaliza strings, elimina duplicados e organiza dataset.
- **Load**: garante schema da tabela `movies`, realiza insercao com controle de duplicidade e gera dump SQL versionado.

In [ ]:
# Setup inicial
from pathlib import Path
import logging
import pandas as pd
import numpy as np

from src.config import load_config
from src.scraper import RPAMovieScraper
from src.invoice_extraction import InvoiceExtractionPipeline
from src.database import MySQLMovieRepository, build_dump_sql
from src.utils import configure_logging, ensure_directories, normalize_movies_df, now_stamp, save_dataframe_outputs

configure_logging(logging.INFO)
config = load_config()
ensure_directories([config.data_dir, config.output_dir, config.sql_dir, config.output_dir / "invoices"])

invoice_result = {}

print("Configuração carregada.")
print(f"Base URL: {config.base_url}")
print(f"Consulta padrão: {config.default_query}")
print(f"Invoice URL: {config.invoice_url}")
print(f"Invoice targets: {config.invoice_target_indices}")
print(f"Headless: {config.headless}")

Configuracao carregada.
Base URL: https://rpachallenge.com/
Consulta padrao: Avengers
Invoice URL: https://rpachallengeocr.azurewebsites.net/
Invoice targets: (2, 4)
Headless: True


## Execução Passo a Passo

### 1. Extract (RPA + Selenium)
Nesta etapa, o robô:

1. Abre a home page.
2. Clica em **Movie Search**.
3. Busca por `Avengers`.
4. Para cada card, expande `card-reveal` e captura a descrição completa.

In [14]:
# EXTRACT
raw_movies = []

try:
    with RPAMovieScraper(
        base_url=config.base_url,
        movie_search_path=config.movie_search_path,
        timeout_seconds=config.timeout_seconds,
        headless=config.headless,
    ) as scraper:
        raw_movies = scraper.run(query=config.default_query)

    print(f"Total de filmes extraidos: {len(raw_movies)}")
except Exception as exc:
    print(f"Falha na etapa EXTRACT: {exc}")
    raise

raw_df = pd.DataFrame(raw_movies)
raw_path = config.data_dir / "movies_raw.csv"
raw_df.to_csv(raw_path, index=False, encoding="utf-8")

display(raw_df.head())
print(f"Arquivo raw salvo em: {raw_path}")

2026-04-18 09:56:07,680 | INFO | src.scraper | WebDriver iniciado
2026-04-18 09:56:10,246 | INFO | src.scraper | Cards encontrados: 3
2026-04-18 09:56:10,331 | INFO | src.scraper | Processando filme 1/3: 'The Avengers'
2026-04-18 09:56:10,331 | INFO | src.scraper | Descricao capturada: 297 caracteres
2026-04-18 09:56:10,418 | INFO | src.scraper | Processando filme 2/3: 'Avengers: Infinity War'
2026-04-18 09:56:10,418 | INFO | src.scraper | Descricao capturada: 490 caracteres
2026-04-18 09:56:10,497 | INFO | src.scraper | Processando filme 3/3: 'Avengers: Endgame'
2026-04-18 09:56:10,497 | INFO | src.scraper | Descricao capturada: 327 caracteres
2026-04-18 09:56:12,657 | INFO | src.scraper | WebDriver finalizado
Total de filmes extraidos: 3


,name,description
0,The Avengers,When an unexpected enemy emerges and threatens...
1,Avengers: Infinity War,As the Avengers and their allies have continue...
2,Avengers: Endgame,After the devastating events of Avengers: Infi...


Arquivo raw salvo em: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\data\movies_raw.csv


### 2. Transform

Regras aplicadas:

- Trim de espaços
- Exclusão de nulos/vazios
- Remoção de duplicados por nome do filme
- Ordenação alfabética para facilitar auditoria

In [15]:
# TRANSFORM
transformed_df = normalize_movies_df(raw_df)

stamp = now_stamp()
outputs = save_dataframe_outputs(
    transformed_df,
    output_dir=config.output_dir,
    base_name=f"movies_avengers_{stamp}",
)

print(f"Linhas antes: {len(raw_df)}")
print(f"Linhas depois: {len(transformed_df)}")
print(f"CSV: {outputs['csv']}")
print(f"JSON: {outputs['json']}")

display(transformed_df.head(10))

Linhas antes: 3
Linhas depois: 3
CSV: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\outputs\movies_avengers_20260418_095615.csv
JSON: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\outputs\movies_avengers_20260418_095615.json


,name,description
0,Avengers: Endgame,After the devastating events of Avengers: Infi...
1,Avengers: Infinity War,As the Avengers and their allies have continue...
2,The Avengers,When an unexpected enemy emerges and threatens...


### 3. Load (MySQL)

Nesta etapa a tabela `movies` é validada/criada e os dados são persistidos com estratégia de `upsert` para evitar duplicidades em reexecuções.

In [ ]:
# LOAD
inserted_rows = 0
load_error = None

try:
    with MySQLMovieRepository(
        host=config.mysql_host,
        port=config.mysql_port,
        database=config.mysql_database,
        user=config.mysql_user,
        password=config.mysql_password,
    ) as repo:
        repo.ensure_schema()
        inserted_rows = repo.upsert_movies(transformed_df.to_dict(orient="records"))
    print(f"Carga concluída. Linhas processadas: {inserted_rows}")
except Exception as exc:
    load_error = str(exc)
    print("Carga no MySQL não executada com sucesso nesta máquina.")
    print(f"Motivo: {load_error}")
    print("Os arquivos de output e o dump SQL continuam disponíveis para validação técnica.")

2026-04-18 09:56:18,313 | INFO | src.database | Conexao MySQL estabelecida
2026-04-18 09:56:18,315 | INFO | src.database | Tabela movies validada
2026-04-18 09:56:18,325 | INFO | src.database | Linhas processadas no banco: 3
2026-04-18 09:56:18,327 | INFO | src.database | Conexao MySQL encerrada
Carga concluida. Linhas processadas: 3


### 4. Geração do Dump SQL

Gera um arquivo SQL versionado contendo:

- DDL da tabela `movies`
- Bloco de inserção dos registros transformados
- Estratégia `ON DUPLICATE KEY UPDATE`

In [17]:
dump_path = build_dump_sql(
    transformed_df.to_dict(orient="records"),
    output_file=config.sql_dir / f"movies_dump_{stamp}.sql",
)

print(f"Dump SQL gerado em: {dump_path}")

2026-04-18 09:56:24,527 | INFO | src.database | Dump SQL salvo em: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\sql\movies_dump_20260418_095615.sql
Dump SQL gerado em: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\sql\movies_dump_20260418_095615.sql


### 5. Extração de invoices

Nesta etapa o robô navega para **Invoice Extraction**, aguarda a tabela `#tableSandbox` e identifica dinamicamente as linhas com índices **2** e **4**.

### 6. Download de arquivos

Com os links extraídos, os arquivos são baixados via código para `outputs/invoices/` com nomenclatura consistente:

- `invoice_2.jpg`
- `invoice_4.jpg`

### 7. Compactação dos dados

Após os downloads, os arquivos são compactados em `outputs/invoices.zip` usando `zipfile`.

In [18]:
# INVOICE EXTRACTION + DOWNLOAD + ZIP
invoice_output_dir = config.output_dir / "invoices"
invoice_zip_path = config.output_dir / "invoices.zip"

with RPAMovieScraper(
    base_url=config.base_url,
    movie_search_path=config.movie_search_path,
    timeout_seconds=config.timeout_seconds,
    headless=config.headless,
) as invoice_scraper:
    invoice_scraper.open_home()
    invoice_scraper.go_to_movie_search()

    invoice_pipeline = InvoiceExtractionPipeline(
        driver=invoice_scraper.driver,
        timeout_seconds=config.timeout_seconds,
        table_id=config.invoice_table_id,
    )

    invoice_result = invoice_pipeline.run(
        invoice_url=config.invoice_url,
        target_indices=config.invoice_target_indices,
        invoices_output_dir=invoice_output_dir,
        zip_output_path=invoice_zip_path,
    )

print("Invoice extraction concluida.")
print(f"Indices selecionados: {invoice_result['selected_indices']}")
for file_path in invoice_result["downloaded_files"]:
    print(f"- Download: {file_path}")
print(f"ZIP final: {invoice_result['zip_file']}")

2026-04-18 09:56:35,620 | INFO | src.scraper | WebDriver iniciado
2026-04-18 09:56:39,085 | INFO | src.invoice_extraction | Pagina de Invoice Extraction carregada: https://rpachallengeocr.azurewebsites.net/
2026-04-18 09:56:39,267 | INFO | src.invoice_extraction | Invoices selecionados para download: [2, 4]
2026-04-18 09:56:41,058 | INFO | src.invoice_extraction | Invoice baixado: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\outputs\invoices\invoice_2.jpg
2026-04-18 09:56:41,496 | INFO | src.invoice_extraction | Invoice baixado: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\outputs\invoices\invoice_4.jpg
2026-04-18 09:56:41,542 | INFO | src.invoice_extraction | Arquivo ZIP gerado: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\outputs\invoices.zip
2026-04-18 09:56:43,813 | INFO | src.scraper | WebDriver finalizado
Invoice extraction concluida.
Indices selecionados: [2, 4]
- Download: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\outputs\invoices\invoice_

## Resultados

Resumo executivo da execução do pipeline.

In [ ]:
invoice_files = invoice_result.get("downloaded_files", []) if isinstance(invoice_result, dict) else []
invoice_zip = invoice_result.get("zip_file", "N/A") if isinstance(invoice_result, dict) else "N/A"

results_df = pd.DataFrame(
    [
        {"metrica": "Registros extraídos (raw)", "valor": len(raw_df)},
        {"metrica": "Registros transformados", "valor": len(transformed_df)},
        {"metrica": "Linhas processadas no load", "valor": inserted_rows},
        {"metrica": "Falha no load", "valor": "Sim" if load_error else "Nao"},
        {"metrica": "Invoices baixados", "valor": len(invoice_files)},
        {"metrica": "ZIP de invoices gerado", "valor": "Sim" if invoice_zip != "N/A" else "Não"},
    ]
)

display(results_df)

print("\nPrincipais artefatos gerados:")
print(f"- Raw CSV: {raw_path}")
print(f"- Output CSV: {outputs['csv']}")
print(f"- Output JSON: {outputs['json']}")
print(f"- Dump SQL: {dump_path}")
for file_path in invoice_files:
    print(f"- Invoice file: {file_path}")
print(f"- Invoices ZIP: {invoice_zip}")

,metrica,valor
0,Registros extraidos (raw),3
1,Registros transformados,3
2,Linhas processadas no load,3
3,Falha no load,Nao
4,Invoices baixados,2
5,ZIP de invoices gerado,Sim



Principais artefatos gerados:
- Raw CSV: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\data\movies_raw.csv
- Output CSV: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\outputs\movies_avengers_20260418_095615.csv
- Output JSON: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\outputs\movies_avengers_20260418_095615.json
- Dump SQL: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\sql\movies_dump_20260418_095615.sql
- Invoice file: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\outputs\invoices\invoice_2.jpg
- Invoice file: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\outputs\invoices\invoice_4.jpg
- Invoices ZIP: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\outputs\invoices.zip


## Conclusão

A solução implementa um pipeline ETL/RPA de ponta a ponta com separacao de responsabilidades, automação robusta e outputs auditáveis.

Pontos de destaque:

- Extração de filmes baseada em seletores estáveis e waits explícitos.
- Captura da descrição completa via interação controlada com `card-reveal`.
- Transformação idempotente com deduplicação.
- Carga preparada para reexecução com `upsert` no MySQL.
- Extensao para **Invoice Extraction** com selecao dinâmica de invoices 2 e 4.
- Download dos arquivos via código e compactação final em `outputs/invoices.zip`.
- Entrega de artefatos técnicos (CSV, JSON, SQL dump e ZIP) para rastreabilidade.

Com pequenos ajustes de infraestrutura (segredos, observabilidade e orquestração), o projeto está pronto para evoluir para ambiente produtivo.